In [ ]:
"""
Pre-claim HbA1c Autoencoder for Diabetic Claims (Scaled for 3M rows)
Detects anomalous lab value patterns (HBA1C, CREATININE, UREA) for diabetes claims
"""

# HbA1c Fraud Detection Model (Scaled for 3M Rows)

This script implements a Per‑Claim Autoencoder for diabetic claims:
- **Model: Per‑Claim Autoencoder** - Detects anomalous lab value patterns
- **Optimized for 3M rows** with increased capacity and larger batch sizes

## Dataset
- Diabetic claims with HbA1c, Creatinine, Urea tests
- Features: Demographics + 3 lab values
- Labels: `is_fraud` (0/1)

## 1. Setup and Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import joblib
import json
from datetime import datetime
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Set random seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ==================== AZURE ML PATHS ====================
# Path.home() resolves to /home/azureuser/ in Azure ML
BASE_DIR = Path.home() / "localfiles" / "lstm"

# Define paths for diabetic dataset
DATA_PATH = BASE_DIR / "data" / "kenyan_diabetic_claims_20260321_145103.csv"
MODEL_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "visualizations" / "plots"

# Create directories
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Validation
print(f"✅ Base Directory: {BASE_DIR}")
print(f"✅ Data Path: {DATA_PATH}")
print(f"✅ Data exists: {DATA_PATH.exists()}")

print("\n✅ Dependencies loaded")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

2026-03-21 14:53:15.502279: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 14:53:19.887450: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 14:53:19.887568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 14:53:20.764050: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 14:53:22.462046: I tensorflow/core/platform/cpu_feature_guar

✅ Base Directory: /home/azureuser/localfiles/lstm
✅ Data Path: /home/azureuser/localfiles/lstm/data/kenyan_diabetic_claims_20260321_145103.csv
✅ Data exists: True

✅ Dependencies loaded
TensorFlow version: 2.15.0
GPU available: []


2026-03-21 14:53:35.031126: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## 2. Data Loading and Exploration

In [2]:
# Load dataset
print("Loading dataset...")
df_full = pd.read_csv(DATA_PATH)
print(f"Full dataset shape: {df_full.shape}")
print(f"Memory usage: {df_full.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display basic info
print("\nDataset Info:")
print(f"Claims: {df_full['claim_id'].nunique():,}")
print(f"Patients: {df_full['patient_id'].nunique():,}")
print(f"Facilities: {df_full['facility_id'].nunique():,}")
print(f"\nFraud rate: {df_full['is_fraud'].mean():.2%}")
print(f"\nDiagnosis distribution:")
print(df_full['diagnosis'].value_counts())

Loading dataset...
Full dataset shape: (3000000, 20)
Memory usage: 2945.03 MB

Dataset Info:
Claims: 3,000,000
Patients: 1,036,454
Facilities: 90,000

Fraud rate: 20.00%

Diagnosis distribution:
Diabetic Nephropathy    1000000
Diabetes - (DKA/HHS)    1000000
Diabetic Infections     1000000
Name: diagnosis, dtype: int64


## 3. Sample Data for Quick Testing

In [3]:
# For quick testing, use a small sample (100 rows for testing)
TEST_MODE = False  # Set to False for full 3M training
SAMPLE_SIZE = min(100, len(df_full)) if TEST_MODE else len(df_full)

if TEST_MODE:
    df = df_full.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"\n🧪 TEST MODE: Using {len(df)} samples for validation")
else:
    df = df_full.copy()
    print(f"\n🚀 FULL TRAINING MODE: Using {len(df):,} samples")
    
# Convert date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nFraud distribution:")
print(df['is_fraud'].value_counts())
print(f"\nDiagnosis distribution:")
print(df['diagnosis'].value_counts())

# Display sample
df[['claim_id', 'facility_name', 'date', 'diagnosis', 'is_fraud', 'HBA1C', 'CREATININE', 'UREA']].head(10)


🚀 FULL TRAINING MODE: Using 3,000,000 samples

Date range: 2022-02-01 00:00:00 to 2026-02-28 00:00:00

Fraud distribution:
0    2400000
1     600000
Name: is_fraud, dtype: int64

Diagnosis distribution:
Diabetic Nephropathy    1000000
Diabetes - (DKA/HHS)    1000000
Diabetic Infections     1000000
Name: diagnosis, dtype: int64


,claim_id,facility_name,date,diagnosis,is_fraud,HBA1C,CREATININE,UREA
0,CLM-2026-1351490,Mwatate Dispensary,2022-02-01,Diabetic Nephropathy,0,12.9,3.1,126.0
1,CLM-2026-1111106,Naivasha Sub-county Hospital,2022-02-01,Diabetes - (DKA/HHS),0,9.7,4.3,112.0
2,CLM-2026-1397604,Lamu County Hospital,2022-02-01,Diabetic Nephropathy,0,11.3,4.6,119.0
3,CLM-2026-1162249,Ongata Rongai Health Centre,2022-02-01,Diabetic Nephropathy,0,8.4,1.2,22.0
4,CLM-2026-153010,Eldoret Health Centre,2022-02-01,Diabetic Nephropathy,0,8.8,2.5,79.0
5,CLM-2026-271423,Kakamega County Referral,2022-02-01,Diabetic Infections,0,10.4,1.4,33.0
6,CLM-2026-242971,Vihiga County Hospital,2022-02-01,Diabetic Infections,0,5.7,0.5,7.9
7,CLM-2026-764716,Namanga Health Centre,2022-02-01,Diabetes - (DKA/HHS),0,13.3,3.3,40.0
8,CLM-2026-1840458,Lodwar County Referral,2022-02-01,Diabetic Nephropathy,0,5.3,1.2,20.6
9,CLM-2026-1117663,Kisii Level 5 Hospital,2022-02-01,Diabetes - (DKA/HHS),0,9.7,0.9,31.0


## 4. Feature Engineering

In [4]:
# ==================== FEATURE DEFINITIONS ====================
# Lab features for diabetes (3 features)
LAB_FEATURES = ['HBA1C', 'CREATININE', 'UREA']
N_LAB_FEATURES = len(LAB_FEATURES)

# Demographic features
DEMO_FEATURES = ['age', 'sex']
# Sex encoding: Male=1, Female=0
df['sex_encoded'] = (df['sex'] == 'Male').astype(int)

# Complete feature set for model
ALL_FEATURES = ['age', 'sex_encoded'] + LAB_FEATURES
N_FEATURES = len(ALL_FEATURES)

print(f"\n📊 Feature set:")
print(f"  - Lab features ({N_LAB_FEATURES}): {LAB_FEATURES}")
print(f"  - Demographic features (2): age, sex")
print(f"  - Total features: {N_FEATURES}")

# Check for missing values
print("\nMissing values:")
print(df[ALL_FEATURES].isnull().sum())

# Handle any missing values
df[ALL_FEATURES] = df[ALL_FEATURES].ffill().bfill()

# Check for outliers/invalid values
print("\n📊 Lab value statistics:")
print(df[LAB_FEATURES].describe())

# Display scaled data sample
print("\nSample raw data:")
df[ALL_FEATURES].head(10)


📊 Feature set:
  - Lab features (3): ['HBA1C', 'CREATININE', 'UREA']
  - Demographic features (2): age, sex
  - Total features: 5

Missing values:
age            0
sex_encoded    0
HBA1C          0
CREATININE     0
UREA           0
dtype: int64

📊 Lab value statistics:
              HBA1C    CREATININE          UREA
count  3.000000e+06  3.000000e+06  3.000000e+06
mean   8.541798e+00  1.644000e+00  4.464026e+01
std    3.902673e+00  1.401391e+00  3.560239e+01
min    1.100000e+00  0.000000e+00  8.000000e-01
25%    5.300000e+00  7.000000e-01  1.620000e+01
50%    9.000000e+00  1.200000e+00  3.800000e+01
75%    1.140000e+01  2.300000e+00  6.400000e+01
max    1.800000e+01  7.500000e+00  1.800000e+02

Sample raw data:


,age,sex_encoded,HBA1C,CREATININE,UREA
0,77,1,12.9,3.1,126.0
1,84,0,9.7,4.3,112.0
2,42,0,11.3,4.6,119.0
3,2,1,8.4,1.2,22.0
4,74,0,8.8,2.5,79.0
5,6,1,10.4,1.4,33.0
6,11,0,5.7,0.5,7.9
7,57,1,13.3,3.3,40.0
8,19,1,5.3,1.2,20.6
9,7,0,9.7,0.9,31.0


## 5. Model: Per‑Claim Autoencoder (Scaled for 3M Rows)

### Objective
- Detect anomalous lab value combinations that deviate from normal patterns
- Input: 5 features (age, sex_encoded, HBA1C, CREATININE, UREA)
- Output: Reconstruction of same 5 features
- Anomaly score = reconstruction error
- **Optimized for 3M rows with increased model capacity**

In [5]:
# ==================== MODEL DATA PREPARATION ====================

# Extract features
X_raw = df[ALL_FEATURES].values
print(f"\nRaw features shape: {X_raw.shape}")

# Scale features to [0, 1] using MinMaxScaler
scaler_model = MinMaxScaler()
X_scaled = scaler_model.fit_transform(X_raw)

# Save scaler
scaler_path = os.path.join(MODEL_DIR, "hba1c_model1_scaler.pkl")
joblib.dump(scaler_model, scaler_path)
print(f"\n✅ Scaler saved to {scaler_path}")

# Display scaled data sample
print("\nScaled data sample (first 5 rows):")
pd.DataFrame(X_scaled[:5], columns=ALL_FEATURES).round(3)


Raw features shape: (3000000, 5)

✅ Scaler saved to /home/azureuser/localfiles/lstm/models/hba1c_model1_scaler.pkl

Scaled data sample (first 5 rows):


,age,sex_encoded,HBA1C,CREATININE,UREA
0,0.77,1.0,0.698,0.413,0.699
1,0.84,0.0,0.509,0.573,0.621
2,0.42,0.0,0.604,0.613,0.660
3,0.02,1.0,0.432,0.160,0.118
4,0.74,0.0,0.456,0.333,0.436


In [6]:
# ==================== TRAIN/VAL/TEST SPLIT ====================

# Split: 70% train, 15% val, 15% test
X_train, X_temp = train_test_split(X_scaled, test_size=0.3, random_state=RANDOM_SEED)
X_val, X_test = train_test_split(X_temp, test_size=0.5, random_state=RANDOM_SEED)

print(f"\nTrain set: {X_train.shape[0]:,} samples")
print(f"Validation set: {X_val.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")


Train set: 2,100,000 samples
Validation set: 450,000 samples
Test set: 450,000 samples


In [8]:
# ==================== BUILD AUTOENCODER ARCHITECTURE (Properly Scaled for 3M Rows) ====================

# Model parameters - significantly increased capacity for 3M rows
LATENT_DIM = 32  # Increased from 12 for richer representation
LEARNING_RATE = 0.001

def build_autoencoder_scaled(input_dim, latent_dim):
    """
    Build a properly scaled autoencoder for 3M rows.
    Features:
    - Much wider layers (128-256-512 range)
    - Deeper architecture for complex pattern learning
    - Progressive compression and expansion
    - Proper regularization to prevent overfitting
    """
    
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,), name='encoder_input')
    
    # Progressive compression
    x = layers.Dense(256, activation='relu', name='encoder_dense1')(encoder_input)
    x = layers.BatchNormalization(name='encoder_bn1')(x)
    x = layers.Dropout(0.3, name='encoder_dropout1')(x)
    
    x = layers.Dense(128, activation='relu', name='encoder_dense2')(x)
    x = layers.BatchNormalization(name='encoder_bn2')(x)
    x = layers.Dropout(0.3, name='encoder_dropout2')(x)
    
    x = layers.Dense(64, activation='relu', name='encoder_dense3')(x)
    x = layers.BatchNormalization(name='encoder_bn3')(x)
    x = layers.Dropout(0.25, name='encoder_dropout3')(x)
    
    x = layers.Dense(48, activation='relu', name='encoder_dense4')(x)
    x = layers.BatchNormalization(name='encoder_bn4')(x)
    x = layers.Dropout(0.25, name='encoder_dropout4')(x)
    
    # Bottleneck
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    
    # Decoder - symmetric expansion
    x = layers.Dense(48, activation='relu', name='decoder_dense1')(latent)
    x = layers.BatchNormalization(name='decoder_bn1')(x)
    x = layers.Dropout(0.25, name='decoder_dropout1')(x)
    
    x = layers.Dense(64, activation='relu', name='decoder_dense2')(x)
    x = layers.BatchNormalization(name='decoder_bn2')(x)
    x = layers.Dropout(0.25, name='decoder_dropout2')(x)
    
    x = layers.Dense(128, activation='relu', name='decoder_dense3')(x)
    x = layers.BatchNormalization(name='decoder_bn3')(x)
    x = layers.Dropout(0.3, name='decoder_dropout3')(x)
    
    x = layers.Dense(256, activation='relu', name='decoder_dense4')(x)
    x = layers.BatchNormalization(name='decoder_bn4')(x)
    x = layers.Dropout(0.3, name='decoder_dropout4')(x)
    
    decoder_output = layers.Dense(input_dim, activation='linear', name='decoder_output')(x)
    
    # Model
    autoencoder = models.Model(encoder_input, decoder_output, name='hba1c_autoencoder_scaled')
    
    return autoencoder

# Build model
model = build_autoencoder_scaled(N_FEATURES, LATENT_DIM)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

model.summary()
print(f"\n📊 Total trainable parameters: {model.count_params():,}")

Model: "hba1c_autoencoder_scaled"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense1 (Dense)          │ (None, 256)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn1                     │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout1 (Dropout)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense2 (Dense)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn2                     │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout2 (Dropout)      │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense3 (Dense)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn3                     │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout3 (Dropout)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense4 (Dense)          │ (None, 48)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn4                     │ (None, 48)             │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout4 (Dropout)      │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense1 (Dense)          │ (None, 48)             │         1,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_bn1                     │ (None, 48)             │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dropout1 (Dropout)      │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense2 (Dense)          │ (None, 64)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_bn2                     │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dropout2 (Dropout)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense3 (Dense)          │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_bn3                     │ (None, 128)            │           51

 Total params: 98,693 (385.52 KB)

 Trainable params: 96,709 (377.77 KB)

 Non-trainable params: 1,984 (7.75 KB)


📊 Total trainable parameters: 98,693


In [10]:
# ==================== TRAINING CONFIGURATION (Optimized for 3M Rows) ====================

# Training parameters - optimized for 3M rows with larger architecture
EPOCHS = 30 if TEST_MODE else 100          # More epochs for convergence with larger model
BATCH_SIZE = 16 if TEST_MODE else 4096     # Increased batch size for 3M rows
PATIENCE = 12 if not TEST_MODE else 5      # More patience for large dataset

# Callbacks
model_path = os.path.join(MODEL_DIR, "hba1c_model1_claim_autoencoder.keras")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        model_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,                         # Increased patience
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(MODEL_DIR, "hba1c_model1_training_log.csv")
    )
]

print("\n🚀 Starting Model training...\n")
print(f"Training Configuration:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE:,}")
print(f"  - Training samples: {X_train.shape[0]:,}")
print(f"  - Validation samples: {X_val.shape[0]:,}")
print(f"  - Early stopping patience: {PATIENCE}")
print(f"  - Learning rate: 0.001")

history = model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

print(f"\n✅ Model training complete!")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")


🚀 Starting Model training...

Training Configuration:
  - Epochs: 100
  - Batch size: 4,096
  - Training samples: 2,100,000
  - Validation samples: 450,000
  - Early stopping patience: 12
  - Learning rate: 0.001
Epoch 1/100


512/513 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.5724 - mae: 0.4742
Epoch 1: val_loss improved from None to 0.01402, saving model to /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder.keras
513/513 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - loss: 0.2136 - mae: 0.2686 - val_loss: 0.0140 - val_mae: 0.0895 - learning_rate: 0.0010
Epoch 2/100
512/513 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0219 - mae: 0.1109
Epoch 2: val_loss improved from 0.01402 to 0.00881, saving model to /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder.keras
513/513 ━━━━━━━━━━━━━━━━━━━━ 32s 63ms/step - loss: 0.0199 - mae: 0.1055 - val_loss: 0.0088 - val_mae: 0.0653 - learning_rate: 0.0010
Epoch 3/100
512/513 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0162 - mae: 0.0950
Epoch 3: val_loss improved from 0.00881 to 0.00823, saving model to /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder.keras
513/513 ━━━━━━━━━━━━━━━━━━━━ 32s 63ms/step - loss: 0.0156 - mae: 0.0

In [11]:
# ==================== PLOT TRAINING HISTORY ====================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
best_epoch = np.argmin(history.history['val_loss'])
ax.scatter(best_epoch, history.history['val_loss'][best_epoch], 
          color='red', s=100, zorder=5, label=f'Best: {history.history["val_loss"][best_epoch]:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('HbA1c Autoencoder: Training and Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# MAE
ax = axes[1]
ax.plot(history.history['mae'], label='Training MAE', linewidth=2)
ax.plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Training and Validation MAE')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save
plot_path = os.path.join(PLOTS_DIR, "hba1c_model1_training_history.png")
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Plot saved to {plot_path}")

✅ Plot saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model1_training_history.png


In [12]:
# ==================== DETERMINE ANOMALY THRESHOLD ====================

# Predict on validation set
X_val_pred = model.predict(X_val, verbose=0)

# Calculate reconstruction errors (MSE per sample)
reconstruction_errors_val = np.mean(np.square(X_val - X_val_pred), axis=1)

# Set threshold at 95th percentile
THRESHOLD_PERCENTILE = 95
threshold = np.percentile(reconstruction_errors_val, THRESHOLD_PERCENTILE)

print(f"\n📊 Reconstruction Error Statistics (Validation):")
print(f"  Mean: {np.mean(reconstruction_errors_val):.6f}")
print(f"  Std:  {np.std(reconstruction_errors_val):.6f}")
print(f"  Min:  {np.min(reconstruction_errors_val):.6f}")
print(f"  Max:  {np.max(reconstruction_errors_val):.6f}")
print(f"\n🎯 Anomaly Threshold ({THRESHOLD_PERCENTILE}th percentile): {threshold:.6f}")


📊 Reconstruction Error Statistics (Validation):
  Mean: 0.000403
  Std:  0.000444
  Min:  0.000003
  Max:  0.006636

🎯 Anomaly Threshold (95th percentile): 0.001247


In [15]:
# ==================== EVALUATE ON TEST SET ====================

# Predict on test set
X_test_pred = model.predict(X_test, verbose=0)

# Calculate reconstruction errors
reconstruction_errors_test = np.mean(np.square(X_test - X_test_pred), axis=1)

# Detect anomalies
is_anomaly = (reconstruction_errors_test > threshold).astype(int)

print(f"\n📊 Test Set Results:")
print(f"  Total samples: {len(X_test):,}")
print(f"  Detected anomalies: {np.sum(is_anomaly):,}")
print(f"  Anomaly rate: {np.mean(is_anomaly):.2%}")
print(f"\n  Mean reconstruction error: {np.mean(reconstruction_errors_test):.6f}")
print(f"  Std reconstruction error:  {np.std(reconstruction_errors_test):.6f}")

# If we have fraud labels, evaluate performance
if 'is_fraud' in df.columns:
    # Get fraud labels for test indices (simplified - need alignment in production)
    print(f"\n📊 Fraud vs Anomaly Detection:")
    print(f"  Note: For production, ensure proper index alignment with test set")
    print(f"  Actual fraud rate in dataset: {df['is_fraud'].mean():.2%}")


📊 Test Set Results:
  Total samples: 450,000
  Detected anomalies: 22,952
  Anomaly rate: 5.10%

  Mean reconstruction error: 0.000405
  Std reconstruction error:  0.000446

📊 Fraud vs Anomaly Detection:
  Note: For production, ensure proper index alignment with test set
  Actual fraud rate in dataset: 20.00%


In [16]:
# ==================== VISUALIZE ANOMALY DISTRIBUTION ====================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram
ax = axes[0, 0]
ax.hist(reconstruction_errors_test, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(threshold, color='red', linewidth=2, linestyle='--', label=f'Threshold: {threshold:.4f}')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Frequency')
ax.set_title('Test Set: Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Box plot
ax = axes[0, 1]
ax.boxplot(reconstruction_errors_test, vert=False)
ax.axvline(threshold, color='red', linewidth=2, linestyle='--')
ax.set_xlabel('Reconstruction Error')
ax.set_title('Box Plot')
ax.grid(True, alpha=0.3)

# 3. Scatter plot (sample for visualization)
ax = axes[1, 0]
sample_size = min(1000, len(reconstruction_errors_test))
colors = ['red' if a else 'green' for a in is_anomaly[:sample_size]]
ax.scatter(range(sample_size), reconstruction_errors_test[:sample_size], c=colors, alpha=0.6, s=20)
ax.axhline(threshold, color='black', linewidth=2, linestyle='--')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Reconstruction Error')
ax.set_title('Anomaly Detection Results (First 1000 samples)')
ax.grid(True, alpha=0.3)

# 4. Feature importance
ax = axes[1, 1]
feature_errors = np.mean(np.square(X_test - X_test_pred), axis=0)
colors = ['red' if e > np.mean(feature_errors) else 'steelblue' for e in feature_errors]
bars = ax.barh(ALL_FEATURES, feature_errors, color=colors, alpha=0.7)
ax.axvline(np.mean(feature_errors), color='black', linestyle='--',
          label=f'Mean: {np.mean(feature_errors):.4f}')
ax.set_xlabel('Average Reconstruction Error')
ax.set_title('Feature-wise Error Contribution')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "hba1c_model1_anomaly_distribution.png")
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Plot saved to {plot_path}")

✅ Plot saved to /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model1_anomaly_distribution.png


In [17]:
# ==================== FEATURE-WISE ERROR ANALYSIS ====================

print("\n📊 Feature-wise Reconstruction Errors:")
for i, feature in enumerate(ALL_FEATURES):
    error = feature_errors[i]
    print(f"  {feature}: {error:.6f}")

# Identify most anomalous features
most_anomalous_idx = np.argmax(feature_errors)
print(f"\n⚠️ Most anomalous feature: {ALL_FEATURES[most_anomalous_idx]}")


📊 Feature-wise Reconstruction Errors:
  age: 0.000282
  sex_encoded: 0.000048
  HBA1C: 0.000322
  CREATININE: 0.000297
  UREA: 0.001074

⚠️ Most anomalous feature: UREA


In [18]:
# ==================== SAVE MODEL METADATA ====================

metadata = {
    'model_name': 'Model: HbA1c Claim Autoencoder (Scaled for 3M)',
    'model_type': 'deep_autoencoder',
    'test_type': 'HBA1C_CREATININE_UREA',
    'input_dim': N_FEATURES,
    'latent_dim': LATENT_DIM,
    'feature_names': ALL_FEATURES,
    'threshold': float(threshold),
    'threshold_percentile': THRESHOLD_PERCENTILE,
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'best_val_loss': float(min(history.history['val_loss'])),
    'test_anomaly_rate': float(np.mean(is_anomaly)),
    'feature_errors': {feature: float(feature_errors[i]) for i, feature in enumerate(ALL_FEATURES)},
    'training_date': datetime.now().isoformat(),
    'epochs_trained': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'test_mode': TEST_MODE
}

# Save metadata
meta_path = os.path.join(MODEL_DIR, "hba1c_model1_claim_autoencoder_meta.json")
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Model saved to: {model_path}")
print(f"✅ Metadata saved to: {meta_path}")
print(f"✅ Scaler saved to: {scaler_path}")


✅ Model saved to: /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder.keras
✅ Metadata saved to: /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder_meta.json
✅ Scaler saved to: /home/azureuser/localfiles/lstm/models/hba1c_model1_scaler.pkl


In [19]:
# ==================== SUMMARY ====================

print("\n" + "="*60)
print("✅✅✅ HBA1C AUTOENCODER TRAINING COMPLETE ✅✅✅")
print("="*60)
print(f"""
Model Summary:
  - Input features: {N_FEATURES} ({ALL_FEATURES})
  - Latent dimension: {LATENT_DIM}
  - Total parameters: {model.count_params():,}
  - Best validation loss: {min(history.history['val_loss']):.6f}
  - Anomaly threshold: {threshold:.6f}
  - Test anomaly rate: {np.mean(is_anomaly):.2%}
  
Files saved:
  - Model: {model_path}
  - Scaler: {scaler_path}
  - Metadata: {meta_path}
  - Training plot: {plot_path}
  
Training configuration:
  - Epochs: {EPOCHS}
  - Batch size: {BATCH_SIZE:,}
  - Learning rate: {LEARNING_RATE}
  - Test mode: {TEST_MODE}
  - Training samples: {X_train.shape[0]:,}
""")


✅✅✅ HBA1C AUTOENCODER TRAINING COMPLETE ✅✅✅

Model Summary:
  - Input features: 5 (['age', 'sex_encoded', 'HBA1C', 'CREATININE', 'UREA'])
  - Latent dimension: 32
  - Total parameters: 98,693
  - Best validation loss: 0.000403
  - Anomaly threshold: 0.001247
  - Test anomaly rate: 5.10%
  
Files saved:
  - Model: /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder.keras
  - Scaler: /home/azureuser/localfiles/lstm/models/hba1c_model1_scaler.pkl
  - Metadata: /home/azureuser/localfiles/lstm/models/hba1c_model1_claim_autoencoder_meta.json
  - Training plot: /home/azureuser/localfiles/lstm/visualizations/plots/hba1c_model1_anomaly_distribution.png
  
Training configuration:
  - Epochs: 100
  - Batch size: 4,096
  - Learning rate: 0.001
  - Test mode: False
  - Training samples: 2,100,000

